# S04_SED-1 lambda-reaction replacement record

This notebook reads canonical total-DOC/CN forcing and writes separate lambda-ready
copies. Baseline files remain total-carbon products; donor-bin datasets are never
appended to them. The notebook configures cyclic spinup and prefire only; postfire
restart XML remains intentionally deferred.


# Add Lambda 10-bin Reaction Sandbox to ATS-flow model xml files

- **Input**: ATS flow xml files from `caseflow-run1` and `caseflow-run2`
- **Output**: ATS-PFLOTRAN xml files with suffix `*.v1.6_pflotran.xml`
- **Scenario**: s1 with dynamic NH4⁺, DOC (10 lambda bins) from ELM
- **Reaction**: S04_SED-1 10-bin lambda-PFLOTRAN medoid network; NSE-selected PFLOTRAN R18 kinetics
- **Species (20)**: HCO3⁻, NH4⁺, HPO4²⁻, HS⁻, H⁺, O2(aq), BIOMASS, C38/C35/C29/C27/C25/C23/C20/C19/C18/C13-DONOR, Tsw, Tgw, Tr
- **DOC forcing**: Total-DOC S04S10 HDF5 inputs, partitioned as equal molC fractions (1/10 per bin) and converted to species units
- **Timestep**: Fixed at 1000s

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
from pathlib import Path
import shutil
import subprocess
import h5py as h5
import matplotlib.pyplot as plt
import re
import glob
import numpy as np

from scipy.io import loadmat

In [ ]:
# a temp solution, need to update lib
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning, message='.*product.*')

In [ ]:
# Parameters cell -- schema-v2 date-based configuration
from config_utils import load_config, phase_dates, phase_label, phase_names, phase_period, noleap_day_of_year, phase_forcing_dir, full_timeline_forcing_dir

config = load_config('config.json')
case = config['case']
watershed_name = case['watershed_name']
hucs = case['hucs']
site_name = case['site_name']
meshsize_nx = case['meshsize_nx']

spinup_dates = phase_dates(config, 'spinup')
prefire_dates = phase_dates(config, 'prefire_transient')
postfire_dates = phase_dates(config, 'postfire_transient') if 'postfire_transient' in config else []
spinup_label = phase_label(config, 'spinup')
prefire_label = phase_label(config, 'prefire_transient')
postfire_label = phase_label(config, 'postfire_transient') if postfire_dates else None
forcing_spinup_dir = phase_forcing_dir(config, 'spinup')
forcing_prefire_dir = phase_forcing_dir(config, 'prefire_transient')
forcing_postfire_dir = phase_forcing_dir(config, 'postfire_transient') if postfire_dates else None
forcing_full_dir = full_timeline_forcing_dir(config)
for _d in (forcing_spinup_dir, forcing_prefire_dir, forcing_postfire_dir, forcing_full_dir):
    if _d is not None: _d.mkdir(parents=True, exist_ok=True)

# Temporary aliases keep unchanged downstream cells executable while their
# date selections use the phase-specific date lists above.
start_year_spinup = spinup_dates[0].year
end_year_spinup = spinup_dates[-1].year
nyears_steadystate_spinup = config['spinup']['steady_state_years']
nyears_cyclic_spinup = config['spinup']['cyclic_years']
start_year_transient = prefire_dates[0].year
end_year_transient = prefire_dates[-1].year
run = config['prefire_transient']['elm_run']


In [ ]:
# check get_docflux_from_ELM_3D.ipynb for interpretation of f_DOM and k
#f_DOM, fdom = 1, '1'
f_DOM, fdom = 0.01, '001'

# k ranges from 0.1~1.5 h^-1, to convert DOC flux to DOC concentration
k_in_sec = np.array([0.1, 1.5])/3600.0
#k, k_label = k_in_sec[0], '01'
k, k_label = k_in_sec[1], '15'

# years of data used to generate the source term and boundary conditions
# noticing the difference to nyears_steadystate_spinup and nyears_cyclic_spinup loaded from config.json
years_spinup    = np.arange(start_year_spinup, end_year_spinup+1)
years_transient = np.arange(start_year_transient, end_year_transient+1)

print(list(years_spinup))
print(list(years_transient))

In [ ]:
m2_mat_filename =  f'../data-processed/{site_name}/m2_coords_{site_name}.mat'
loaded_data = loadmat(m2_mat_filename)
meshsize_nx = loaded_data['meshsize_nx'].flatten()[0]

flag_scenario = 's1'
outputs = {}

dzs_soil  = loaded_data['dzs_soil'].flatten()
dzs_geo   = loaded_data['dzs_geo'].flatten()
meshsize_nz = len(dzs_soil) + len(dzs_geo)

In [ ]:
# Add atspflotranutils to sys.path
notebooks_dir = Path().resolve()
if notebooks_dir.name != 'notebooks' or not (notebooks_dir / 'config.json').is_file():
    raise RuntimeError(
        f'Run this notebook from its notebooks/ directory; current directory: {notebooks_dir}'
    )
base_dir = notebooks_dir.parent
utils_path = base_dir / 'atspflotranutils'
sys.path.append(str(utils_path))

# Check if the path was added successfully
print("Paths in sys.path:")
print("\n".join(sys.path))

In [ ]:
# ELM total DOC/CN forcing is generated in its baseline phase directories; no files are copied here.


In [ ]:
# Baseline total-DOC/CN inputs and separate lambda-ready outputs.
spinup_doc = forcing_spinup_dir / f'{site_name}_DOC_source_cyclic{nyears_cyclic_spinup}y.h5'
spinup_cnbc = forcing_spinup_dir / f'{site_name}_CNbc_conc_cyclic{nyears_cyclic_spinup}y.h5'
full_doc = forcing_full_dir / f'{site_name}_DOC_source.h5'
full_cnbc = forcing_full_dir / f'{site_name}_CNbc_conc.h5'
lambda_spinup_dir = forcing_spinup_dir / 'lambda'
lambda_full_dir = forcing_full_dir / 'lambda'
lambda_spinup_dir.mkdir(parents=True, exist_ok=True)
lambda_full_dir.mkdir(parents=True, exist_ok=True)
lambda_spinup_doc = lambda_spinup_dir / f'{site_name}_DOC_source_lambda10.h5'
lambda_spinup_cnbc = lambda_spinup_dir / f'{site_name}_CNbc_conc_lambda10.h5'
lambda_full_doc = lambda_full_dir / f'{site_name}_DOC_source_lambda10.h5'
lambda_full_cnbc = lambda_full_dir / f'{site_name}_CNbc_conc_lambda10.h5'


In [ ]:
n_lambda_bins = 10
bin_species = ['C38-DONOR', 'C35-DONOR', 'C29-DONOR', 'C27-DONOR', 'C25-DONOR',
               'C23-DONOR', 'C20-DONOR', 'C19-DONOR', 'C18-DONOR', 'C13-DONOR']
c_per_bin = [38.92, 35.42, 29.79591836734694, 27.7, 25.79591836734694,
             23.66, 20.52, 19.632653061224488, 18.183673469387756, 13.22]

def copy_with_lambda_bins(total_doc, total_cnbc, lambda_doc, lambda_cnbc):
    """Copy total forcing unchanged, then add S04 donor-bin fields to the copy."""
    shutil.copy2(total_doc, lambda_doc)
    shutil.copy2(total_cnbc, lambda_cnbc)
    with h5.File(lambda_doc, 'a') as hdf:
        total = hdf['DOC production [molC m^-3 s^-1]']
        for species, c_atoms in zip(bin_species, c_per_bin):
            key = f'DOC production {species} [mol m^-3 s^-1]'
            if key in hdf: del hdf[key]
            group = hdf.create_group(key)
            for i in range(len(total)):
                group.create_dataset(str(i), data=total[str(i)][:] / n_lambda_bins / c_atoms)
    with h5.File(lambda_cnbc, 'a') as hdf:
        total = hdf['DOC mol water basis v1 [molS molH^-1]'][:]
        for species, c_atoms in zip(bin_species, c_per_bin):
            key = f'{species} mol water basis [molS molH^-1]'
            if key in hdf: del hdf[key]
            hdf.create_dataset(key, data=total / n_lambda_bins / c_atoms)

def verify_lambda_bins(lambda_doc, lambda_cnbc):
    with h5.File(lambda_doc, 'r') as hdf:
        total = hdf['DOC production [molC m^-3 s^-1]']
        for i in range(len(total)):
            recovered = sum(hdf[f'DOC production {species} [mol m^-3 s^-1]'][str(i)][:] * atoms for species, atoms in zip(bin_species, c_per_bin))
            if not np.allclose(recovered, total[str(i)][:]): raise ValueError(f'DOC donor-bin carbon check failed at sample {i}')
    with h5.File(lambda_cnbc, 'r') as hdf:
        recovered = sum(hdf[f'{species} mol water basis [molS molH^-1]'][:] * atoms for species, atoms in zip(bin_species, c_per_bin))
        if not np.allclose(recovered, hdf['DOC mol water basis v1 [molS molH^-1]'][:]): raise ValueError('CN boundary donor-bin carbon check failed')

for required in (spinup_doc, spinup_cnbc, full_doc, full_cnbc):
    if not required.exists(): raise FileNotFoundError(f'Run get_docflux_from_ELM_3D.ipynb first: {required}')
copy_with_lambda_bins(spinup_doc, spinup_cnbc, lambda_spinup_doc, lambda_spinup_cnbc)
copy_with_lambda_bins(full_doc, full_cnbc, lambda_full_doc, lambda_full_cnbc)
verify_lambda_bins(lambda_spinup_doc, lambda_spinup_cnbc)
verify_lambda_bins(lambda_full_doc, lambda_full_cnbc)
print(f'Wrote verified lambda forcing: {lambda_spinup_dir} and {lambda_full_dir}')


# plot DOC injection flux and BC concentrations

In [ ]:
# Replaced by non-destructive lambda conversion in cell 11.


In [ ]:
# Replaced by non-destructive lambda conversion in cell 11.


## Merge and Create hdf5 files

In [ ]:
# Replaced by non-destructive lambda conversion in cell 11.


In [ ]:
# Replaced by non-destructive lambda conversion in cell 11.


In [ ]:
# Replaced by non-destructive lambda conversion in cell 11.


# pflotranate atsflow xml to atspflotran xml

In [ ]:
from pflotranate_2d_lambda import pflotranate
pflotranate_path = base_dir / 'atspflotranutils' / 'pflotranate_2d_lambda'

import importlib
importlib.reload(pflotranate)

In [ ]:
# Prepare lambda spinup and explicitly named prefire folders.
source_path_run1 = base_dir / 'caseflow-run1' / f'{site_name}_nx{meshsize_nx}_nz{meshsize_nz}.run1.v1.6.xml'
target_folder_run1 = base_dir / f'caselambda-run1.{flag_scenario}'; target_path_run1 = target_folder_run1 / f'{site_name}_nx{meshsize_nx}_nz{meshsize_nz}.run1.v1.6.xml'; os.makedirs(target_folder_run1, exist_ok=True); shutil.copy(source_path_run1, target_path_run1)
target_folder_run2 = base_dir / f'caselambda-run2-prefire.{flag_scenario}'; os.makedirs(target_folder_run2, exist_ok=True)
for folder in (target_folder_run1, target_folder_run2): shutil.copytree(pflotranate_path / 'reactions', folder / 'reactions', dirs_exist_ok=True)


In [ ]:
# Simulate command-line arguments
command = f"python {pflotranate_path / 'pflotranate.py'} {target_path_run1}"

try:
    subprocess.run(command, shell=True, check=True)
    #print(f"Successfully executed: {command}")
except subprocess.CalledProcessError as e:
    print(f"Error occurred: {e}")

# modify the xml file for caselambda-run1

In [ ]:
# amanzi_xml, included in AMANZI_SRC_DIR/tools/amanzi_xml
import amanzi_xml.utils.io as aio
import amanzi_xml.utils.search as asearch
import amanzi_xml.utils.errors as aerrors
from amanzi_xml.common.parameter import Parameter
from amanzi_xml.common.parameter_list import ParameterList

In [ ]:
xml_filename = base_dir / f'caselambda-run1.{flag_scenario}' / f'{site_name}_nx{meshsize_nx}_nz{meshsize_nz}.run1.v1.6_pflotran.xml'
xml = aio.fromFile(xml_filename)

## revise the initial conditions for flow part

In [ ]:
caseflow_run1_checkpoint = f'../../caseflow-run1/{site_name}/checkpoint_final.h5'

subsurface_flow_IC = asearch.find_path(xml, ['PKs', 'subsurface flow', 'initial conditions', 'restart file'])
subsurface_flow_IC.set("value", caseflow_run1_checkpoint)
print(subsurface_flow_IC)
aio.toFile(xml, xml_filename)

## revise the soil domain info used for DOC injection source terms

In [ ]:
outputs['soil_region_string'] = f'../data-processed/{site_name}/soil_region.txt'
with open(outputs['soil_region_string'], 'r') as f:
    loaded_string = f.read().strip()
soil_domain_region = f"{{{loaded_string}}}"

# Update regions for all 10 DOC source bins
for i in range(1, n_lambda_bins + 1):
    bin_name = f'DOC production bin{i}'
    subsurface_mass_source = asearch.find_path(xml, ['PKs', 'source terms', 'component mass source', bin_name, 'regions'])
    subsurface_mass_source.set("value", soil_domain_region)
    print(f"  Updated regions for '{bin_name}'")

aio.toFile(xml, xml_filename)

## revise the DOC injection hdf5 file info

In [ ]:
# Cyclic spinup reads the separate lambda-ready copy, with a phase-local zero clock.
elm_filename = '../' + str(lambda_spinup_doc)
for i in range(1, n_lambda_bins + 1):
    asearch.find_path(xml, ['PKs', 'source terms', 'component mass source', f'DOC production bin{i}', 'source function', 'file']).set('value', elm_filename)
aio.toFile(xml, xml_filename)


## revise the CN boundary conditions hdf5 file info

In [ ]:
# Cyclic spinup reads the separate lambda-ready CN boundary copy.
elm_filename = '../' + str(lambda_spinup_cnbc)
bc_locations = [('subsurface transport', 'left_subsurface bc'), ('subsurface transport', 'right_subsurface bc'), ('surface transport', 'left_surface bc'), ('surface transport', 'right_surface bc')]
for pk_name, bc_name in bc_locations:
    for dof_idx in [2, *range(8, 18)]:
        asearch.find_path(xml, ['PKs', pk_name, 'boundary conditions', 'mole fraction', bc_name, 'boundary mole fraction function', f'dof {dof_idx} function', 'function-tabular', 'file']).set('value', elm_filename)
aio.toFile(xml, xml_filename)


## revise other configurations specifically for caselambda

In [ ]:
## cycle driver
cycle_driver = asearch.find_path(xml, ['cycle driver', 'end time'])
cycle_driver.set("value", str(nyears_cyclic_spinup*365))
aio.toFile(xml, xml_filename)

cycle_driver = asearch.find_path(xml, ['cycle driver', 'end cycle'])
cycle_driver.set("value", "10000000") # 100,000 -> 166.4day, so for 3650day, change to 1e7
aio.toFile(xml, xml_filename)

In [ ]:
## visualization
visualization = asearch.find_path(xml, ['visualization', 'domain', 'times start period stop'])
visualization.set("value", "{0.0, 1.0, -1.0}")
aio.toFile(xml, xml_filename)

visualization = asearch.find_path(xml, ['visualization', 'domain', 'time units'])
visualization.set("value", "d")
aio.toFile(xml, xml_filename)

visualization = asearch.find_path(xml, ['visualization', 'surface', 'times start period stop'])
visualization.set("value", "{0.0, 1.0, -1.0}")
aio.toFile(xml, xml_filename)

visualization = asearch.find_path(xml, ['visualization', 'surface', 'time units'])
visualization.set("value", "d")
aio.toFile(xml, xml_filename)

In [ ]:
## checkpoint
checkpoint = asearch.find_path(xml, ['checkpoint'])
pl1a = Parameter(name="times start period stop", ptype="Array(double)", value="{0, 0.1, -1}")
pl1b = Parameter(name="times start period stop units", ptype="string", value="y")
pl1c = Parameter(name="file name digits", ptype="int", value="7") #should be enough for 10y sim.
checkpoint.append(pl1a)
checkpoint.append(pl1b)
checkpoint.append(pl1c)
aio.toFile(xml, xml_filename)

## revise initial conditions set in file pflotran_chemistry_lambda10bin_pyc.txt
- this is only for the lambda spinup run1

In [ ]:
# Load total fields from the lambda-ready spinup copy for the existing IC diagnostic.
with h5.File(lambda_spinup_cnbc, 'r') as f:
    time_spinup_bc = f['Time'][:]
    nh4_conc_spinup_bc = f['NH4+ mol water basis [molS molH^-1]'][:]
    doc_conc_spinup_bc = f['DOC mol water basis v1 [molS molH^-1]'][:]


In [ ]:
## plot and calculate averaged values
fig, axs = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axs[0].plot(time_spinup_bc, nh4_conc_spinup_bc, color='b')
axs[0].set_ylabel('NH4+ [molS/molH]')
axs[0].set_title('NH4+ Concentration Over Time')

axs[1].plot(time_spinup_bc, doc_conc_spinup_bc, color='r')
axs[1].set_ylabel('DOC [molS/molH]')
axs[1].set_title('DOC Concentration Over Time')
axs[1].set_xlabel('Time [s]')

plt.tight_layout()
plt.show()

In [ ]:
## calculate the average values
avg_nh4 = nh4_conc_spinup_bc.mean()
avg_doc = doc_conc_spinup_bc.mean()

## convert unit from mol/molH to mol/L (PFLOTRAN uses mol/L)
rho_m = 55000.  # molH/m3, water molar density
avg_nh4_molS_L = avg_nh4 * rho_m / 1000  # molS/molH -> molS/L
avg_doc_molS_L = avg_doc * rho_m / 1000  # molC/molH -> molC/L (total DOC in carbon basis)

print(f"Time-averaged NH4+ concentration: {avg_nh4_molS_L:.3e} [molS/L]")
print(f"Time-averaged DOC concentration (total): {avg_doc_molS_L:.3e} [molC/L]")

## manual config (optional override)
avg_doc_molS_L = 2.0e-4  # total DOC in molC/L
print(f"\nUsing manual total DOC = {avg_doc_molS_L:.3e} [molC/L]")
print(f"  Per bin (equal molC): {avg_doc_molS_L / n_lambda_bins:.3e} [molC/L]")
print(f"  Per species example (C38-DONOR): {avg_doc_molS_L / n_lambda_bins / 38.92:.3e} [mol_species/L]")
print(f"  Per species example (C13-DONOR): {avg_doc_molS_L / n_lambda_bins / 13.22:.3e} [mol_species/L]")

In [ ]:
## revise file pflotran_chemistry_lambda10bin_pyc.txt
## for lambda spinup run1: set initial conditions for all 10 bins + NH4+
## PFLOTRAN uses species concentration: species_conc = molC_per_bin / C_i
def replace_species(text, constraint_name, replacements):
    pattern = re.compile(
        rf"(CONSTRAINT {constraint_name}\s+CONCENTRATIONS\s+)(.*?)(/)",
        re.DOTALL
    )
    def repl(match):
        block = match.group(2)
        for species, value in replacements.items():
            block = re.sub(
                rf"({re.escape(species)}\s+)[\deE\.\-]+",
                lambda m: m.group(1) + f"{value:.5e}",
                block
            )
        return match.group(1) + block + match.group(3)
    return pattern.sub(repl, text)

file_path = target_folder_run1 / 'reactions/pflotran_chemistry_lambda10bin_pyc.txt'

# Build replacements dict: all 10 DOC bins (per-species) + NH4+
carbon_species = ['C38-DONOR', 'C35-DONOR', 'C29-DONOR', 'C27-DONOR', 'C25-DONOR',
                  'C23-DONOR', 'C20-DONOR', 'C19-DONOR', 'C18-DONOR', 'C13-DONOR']
c_per_bin_local = [38.92, 35.42, 29.79591836734694, 27.7, 25.79591836734694,
                   23.66, 20.52, 19.632653061224488, 18.183673469387756, 13.22]

replacements = {"NH4+": avg_nh4_molS_L}
# Equal molC per bin, then convert to species concentration
avg_doc_perbin_molC_L = avg_doc_molS_L / n_lambda_bins  # molC/L per bin (equal distribution)
for sp, c_atoms in zip(carbon_species, c_per_bin_local):
    replacements[sp] = avg_doc_perbin_molC_L / c_atoms  # species concentration

with open(file_path, "r") as f:
    content = f.read()

content = replace_species(content, "ICsubsurface", replacements)
content = replace_species(content, "ICsurface", replacements)

with open(file_path, "w") as f:
    f.write(content)

print(f"Updated {file_path.name}:")
print(f"  NH4+ = {avg_nh4_molS_L:.5e}")
print(f"  Total DOC = {avg_doc_molS_L:.3e} molC/L")
print(f"  Per bin molC = {avg_doc_perbin_molC_L:.3e} molC/L")
for sp, c_atoms in zip(carbon_species, c_per_bin_local):
    print(f"  {sp}: {avg_doc_perbin_molC_L / c_atoms:.5e} mol_species/L (÷{c_atoms})")

# modify the xml file for caselambda-run2

In [ ]:
source_path_run2 = base_dir / f'caselambda-run1.{flag_scenario}' / f'{site_name}_nx{meshsize_nx}_nz{meshsize_nz}.run1.v1.6_pflotran.xml'
target_folder_run2 = base_dir / f'caselambda-run2-prefire.{flag_scenario}'; target_path_run2 = target_folder_run2 / f'{site_name}_nx{meshsize_nx}_nz{meshsize_nz}.run2.v1.6_pflotran.xml'; shutil.copy(source_path_run2, target_path_run2)


In [ ]:
xml_filename = base_dir / f'caselambda-run2-prefire.{flag_scenario}' / f'{site_name}_nx{meshsize_nx}_nz{meshsize_nz}.run2.v1.6_pflotran.xml'
xml = aio.fromFile(xml_filename)

# Reactive prefire continues from the final reactive cyclic-spinup checkpoint.
for node in asearch.findall_path(xml, ['initial condition', 'restart file']):
    node.set('value', f'../../caselambda-run1.{flag_scenario}/{site_name}/checkpoint_final.h5')
aio.toFile(xml, xml_filename)


## ~~revise the initial condition for flow part~~

## ~~revise the soil domain info used for DOC injection source terms~~

## revise the water_head/Daymet/MODIS_LAI hdf5 file info for flow part

In [ ]:
for domain, region, filename in [('subsurface flow', 'uphill', forcing_full_dir / 'startpt_head.h5'), ('subsurface flow', 'outlet', forcing_full_dir / 'endpt_head.h5'), ('surface flow', 'uphill', forcing_full_dir / 'startpt_head.h5'), ('surface flow', 'outlet', forcing_full_dir / 'endpt_head.h5')]:
    asearch.find_path(xml, ['PKs', domain, 'boundary conditions', 'head', region, 'boundary head', 'function-tabular', 'file']).set('value', '../' + str(filename))
aio.toFile(xml, xml_filename)


In [ ]:
for evaluator in ['surface-incoming_shortwave_radiation', 'surface-precipitation_rain', 'snow-precipitation', 'surface-vapor_pressure_air', 'surface-air_temperature', 'surface-temperature']:
    asearch.find_path(xml, ['state', 'evaluators', evaluator, 'function', 'surface domain', 'function', 'file']).set('value', '../' + str(forcing_full_dir / f'{site_name}_DayMet.h5'))
aio.toFile(xml, xml_filename)


In [ ]:
## MODIS_LAI hdf5 file
## first, load nlcd_labels
outputs['nlcd_labels_string'] = f'../data-processed/{site_name}/nlcd_labels.txt'
with open(outputs['nlcd_labels_string'], 'r') as f:
    nlcd_labels = f.read().splitlines()
print(nlcd_labels)

# Filter out 'Other' first
labels_to_process = [label for label in nlcd_labels if label != 'Other']

In [ ]:
for label in labels_to_process:
    asearch.find_path(xml, ['state', 'evaluators', 'canopy-leaf_area_index', 'function', label, 'function', 'file']).set('value', '../' + str(forcing_full_dir / f'{site_name}_MODIS_LAI.h5'))
aio.toFile(xml, xml_filename)


## revise the DOC injection hdf5 file info

In [ ]:
# Prefire uses the canonical full timeline, whose clock begins at day zero.
elm_filename = '../' + str(lambda_full_doc)
for i in range(1, n_lambda_bins + 1):
    asearch.find_path(xml, ['PKs', 'source terms', 'component mass source', f'DOC production bin{i}', 'source function', 'file']).set('value', elm_filename)
aio.toFile(xml, xml_filename)


## revise the CN boundary conditions hdf5 file info

In [ ]:
# Prefire uses the canonical full timeline lambda-ready CN boundary forcing.
elm_filename = '../' + str(lambda_full_cnbc)
bc_locations = [('subsurface transport', 'left_subsurface bc'), ('subsurface transport', 'right_subsurface bc'), ('surface transport', 'left_surface bc'), ('surface transport', 'right_surface bc')]
for pk_name, bc_name in bc_locations:
    for dof_idx in [2, *range(8, 18)]:
        asearch.find_path(xml, ['PKs', pk_name, 'boundary conditions', 'mole fraction', bc_name, 'boundary mole fraction function', f'dof {dof_idx} function', 'function-tabular', 'file']).set('value', elm_filename)
aio.toFile(xml, xml_filename)


## revise other configurations specifically for caselambda

In [ ]:
prefire_start_day = nyears_cyclic_spinup * 365; prefire_end_day = prefire_start_day + len(prefire_dates)
for name, value in [('start time', prefire_start_day), ('end time', prefire_end_day)]: asearch.find_path(xml, ['cycle driver', name]).set('value', str(value))
asearch.find_path(xml, ['cycle driver', 'end cycle']).set('value', '20000000')
aio.toFile(xml, xml_filename); print(f'Configured caselambda-run2-prefire: {prefire_start_day} to {prefire_end_day} d')


In [ ]:
# ## checkpoint for caselambda run2
# checkpoint = asearch.find_path(xml, ['checkpoint'])
# pl1a = Parameter(name="times start period stop", ptype="Array(double)", value="{0, 0.1, -1}")
# pl1b = Parameter(name="times start period stop units", ptype="string", value="y")
# pl1c = Parameter(name="file name digits", ptype="int", value="7") #should be enough for 10y sim.
# checkpoint.append(pl1a)
# checkpoint.append(pl1b)
# checkpoint.append(pl1c)
# aio.toFile(xml, xml_filename)